# Straight Waveguide Mode-Source Simulation

This notebook is a compact, single-run version of the straight-waveguide setup from the benchmark notebook. It builds one silicon strip waveguide on silica, solves a TE-like mode source, launches that mode in the `+x` direction, shows the geometry, runs the simulation once, and then plots the strongest `Ey` slice captured by a `FieldRecorder`.

The intent is readability rather than benchmark infrastructure. The timing summary reports setup, mode solve, public compile setup, and the combined JIT-plus-run wall time without reaching into BEAMZ's private compiler internals.


## Imports

Load BeamZ from this local checkout, then NumPy, Matplotlib, and small display/timing utilities used throughout the notebook. The plotting DPI is set once so the diagnostic figures render clearly inline.


In [ ]:
# Standard-library utilities for timing and readable output.
import json
import os
import time
from pathlib import Path

# Numerical, plotting, and BeamZ APIs used by the simulation.
import matplotlib.pyplot as plt
import numpy as np
try:
    from IPython.display import display
except ImportError:
    display = print

import beamz as bz
from beamz.analysis import mode_data_to_dataframe, s_parameters

test_mode = os.environ.get("BEAMZ_DOCS_TEST") == "1"

# Use a slightly sharper default for inline notebook figures.
plt.rcParams.update({"figure.dpi": 120})


## Physical Setup

Define the waveguide dimensions, materials, source pulse, grid, CPML boundaries, and two modal DFT monitors for S-parameter diagnostics. A `FieldRecorder` captures sparse time-domain `Ey` frames on the waveguide's horizontal center plane.

The mode source is placed just inside the left CPML and launches explicitly in `+x`. The source transverse plane is requested to be 75% of the previous area, then clamped to the non-PML y/z aperture so the plane does not overlap the transverse CPML regions.


In [ ]:
# Use BeamZ microns as the notebook's physical length unit.
um = bz.um

# Define a simple straight silicon strip waveguide on silica.
waveguide_length = 5.0 * um
core_width = 0.445 * um
core_height = 0.220 * um
port_extension = 1.0 * um
background_ext = 0.1 * um

# Build the full simulation domain from the waveguide and surrounding material extents.
full_width = core_width + 2 * port_extension
substrate_height = port_extension
cladding_height = core_height + port_extension
sim_size = (waveguide_length, full_width, substrate_height + cladding_height + background_ext)

# Define the three materials used in the model.
mat_air = bz.Material(permittivity=1.0)
mat_si = bz.Material(permittivity=3.48**2)
mat_sio2 = bz.Material(permittivity=1.45**2)

# Set the optical pulse and configure a ModeSource preview for the fundamental TE-like mode.
lambda0 = 1.55 * um
freq0 = bz.LIGHT_SPEED / lambda0
source_fwidth = freq0 / 10
source_time = bz.GaussianPulse(
    freq0=freq0,
    fwidth=source_fwidth,
    offset=0.5 if test_mode else 4.0,
)
mode_spec = bz.ModeSpec(num_modes=1, target_neff=0.98 * 3.48, polarization="te")
sparam_freqs = np.linspace(freq0 - 0.5 * source_fwidth, freq0 + 0.5 * source_fwidth, 3 if test_mode else 11)
sparam_wavelengths_um = bz.LIGHT_SPEED / sparam_freqs / um

# Keep the simulation settings intentionally compact for quick notebook iteration.
steps_per_wavelength = 4 if test_mode else 15
run_time = (6 if test_mode else 120) / freq0
max_material_index = np.sqrt(max(mat_air.permittivity, mat_si.permittivity, mat_sio2.permittivity))
grid_resolution = lambda0 / (steps_per_wavelength * max_material_index)
grid_spec = bz.GridSpec.uniform(grid_resolution)
cpml_cells = 4 if test_mode else 10
pml_t = cpml_cells * grid_resolution
progress = not test_mode
non_pml_x_min = -0.5 * sim_size[0] + pml_t
non_pml_x_max = 0.5 * sim_size[0] - pml_t

# Place the source and modal diagnostic planes in the non-PML region.
src_x = non_pml_x_min + 0.75 * um
diagnostic_plane_gap = 0.50 * um
source_back_x = max(src_x - diagnostic_plane_gap, non_pml_x_min + 0.2 * um)
source_forward_x = min(src_x + diagnostic_plane_gap, non_pml_x_max - 0.2 * um)

# Construct the material design: silica below, silicon core centered from z=0 to z=core_height.
design = bz.Design(background=mat_air)
design += bz.Box(
    center=(0, 0, -0.5 * sim_size[2]),
    size=(bz.inf, bz.inf, sim_size[2]),
    material=mat_sio2,
)
design += bz.Box(
    center=(0, 0, 0.5 * core_height),
    size=(bz.inf, core_width, core_height),
    material=mat_si,
)

# Shrink the mode and monitor planes to 75% of the previous area, then clamp them away from transverse CPMLs.
previous_source_area_fraction = 0.65
requested_source_area_fraction = previous_source_area_fraction * 0.75
requested_source_plane_scale = np.sqrt(requested_source_area_fraction)
non_pml_y = max(sim_size[1] - 2 * pml_t, 0.0)
non_pml_z = max(sim_size[2] - 2 * pml_t, 0.0)
source_plane_y = min(sim_size[1] * requested_source_plane_scale, non_pml_y)
source_plane_z = min(sim_size[2] * requested_source_plane_scale, non_pml_z)
source_plane_size = (0.0, source_plane_y, source_plane_z)
source_plane_area_fraction = (source_plane_y * source_plane_z) / (sim_size[1] * sim_size[2])
source_overlaps_transverse_pml = source_plane_y > non_pml_y or source_plane_z > non_pml_z
src_plane = bz.Box(center=(src_x, 0, 0.0), size=source_plane_size)

# Modal diagnostics: one plane behind the source and one just downstream of it.
modal_monitor_common = dict(
    size=source_plane_size,
    freqs=sparam_freqs,
    mode_spec=mode_spec,
)
source_back_monitor = bz.ModeMonitor(
    center=(source_back_x, 0, 0.0),
    name="source_back",
    **modal_monitor_common,
)
source_forward_monitor = bz.ModeMonitor(
    center=(source_forward_x, 0, 0.0),
    name="source_forward",
    **modal_monitor_common,
)

# Create the simulation without sources first so the native mode preview sees the same grid and materials.
t0 = time.perf_counter()
sim0 = bz.Simulation(
    domain=sim_size,
    grid_spec=grid_spec,
    design=design,
    sources=[],
    monitors=[source_back_monitor, source_forward_monitor],
    boundaries=(
        bz.PML(edges=["left", "right"], thickness=pml_t, formulation="cpml"),
        bz.PML(
            edges=["bottom", "top", "front", "back"],
            thickness=pml_t,
            formulation="cpml",
        ),
    ),
    run_time=run_time,
)
setup_wall_s = time.perf_counter() - t0

# Keep the same workflow shape as before: configure the mode source,
# solve/inspect the local mode, then attach the source to the simulation.
# The immutable ModeSource config is both the preview request and the compile-time source spec.
t0 = time.perf_counter()
mode_source_request = bz.ModeSource(
    center=src_plane.center,
    size=src_plane.size,
    direction="+",
    source_time=source_time,
    mode_spec=mode_spec,
)
modes = mode_source_request.solve_modes(sim0, freqs=[freq0])
assert np.all(np.isfinite(np.asarray(modes.neffs)))
mode_source = mode_source_request

# Configure a sparse time-domain field recorder on the horizontal core plane.
field_x_min = sim0.coordinate_offset[0] - 0.5 * sim_size[0]
field_x_max = sim0.coordinate_offset[0] + 0.5 * sim_size[0]
field_z = sim0.coordinate_offset[2] + 0.5 * core_height
field_x_min_rel = (field_x_min - sim0.coordinate_offset[0]) / um
field_x_max_rel = (field_x_max - sim0.coordinate_offset[0]) / um
field_z_rel = (field_z - sim0.coordinate_offset[2]) / um
timed_steps = sim0.num_steps
raw_field_target_frames = min(10, timed_steps)
raw_field_record_interval = max(1, int(np.ceil(timed_steps / raw_field_target_frames)))
field_recorder = bz.FieldRecorder(
    components=("Ey",),
    interval=raw_field_record_interval,
    name="ey_slice",
    center=(0.0, 0.0, 0.5 * core_height),
    size=(sim_size[0], sim_size[1], 0.0),
)
mode_solve_wall_s = time.perf_counter() - t0

# Attach the source and recorder through one immutable Simulation update.
sim = sim0.updated_copy(
    sources=(mode_source,),
    monitors=(source_back_monitor, source_forward_monitor, field_recorder),
)

# Resolve public request metadata for the grid summary.
shape_zyx = tuple(int(v) for v in sim.to_request(num_steps=1).materials.shape)
shape_xyz = (shape_zyx[2], shape_zyx[1], shape_zyx[0])
cells = int(np.prod(shape_zyx))

# Print a quick sanity summary before plotting or running the simulation.
print(f"domain = {[v / um for v in sim_size]} um")
print(f"grid xyz = {shape_xyz}")
print(f"cells = {cells:,}")
print(f"steps = {sim.num_steps:,}")
monitors_by_name = {monitor.name: monitor for monitor in sim.monitors}
source_x_rel = (sim.sources[0].center[0] - sim.coordinate_offset[0]) / um
source_back_x_rel = (monitors_by_name["source_back"].center[0] - sim.coordinate_offset[0]) / um
source_forward_x_rel = (monitors_by_name["source_forward"].center[0] - sim.coordinate_offset[0]) / um
print(f"sources = {len(sim.sources)}, monitors = {len(sim.monitors)}")
print(f"CPML width = {cpml_cells} cells ({pml_t / um:.3f} um) on every side")
print(f"physical x window = {non_pml_x_min / um:.3f} to {non_pml_x_max / um:.3f} um")
print(f"raw field slice z = {field_z_rel:.3f} um")
print(f"raw field slice x = {field_x_min_rel:.3f} to {field_x_max_rel:.3f} um")
print(f"source x = {source_x_rel:.3f} um, direction = {sim.sources[0].direction}")
print(f"source-back monitor x = {source_back_x_rel:.3f} um")
print(f"source-forward monitor x = {source_forward_x_rel:.3f} um")
print(f"source plane yz = {source_plane_y / um:.3f} x {source_plane_z / um:.3f} um")
print(f"S-parameter wavelengths = {sparam_wavelengths_um.min():.3f} to {sparam_wavelengths_um.max():.3f} um ({sparam_freqs.size} points)")
print(f"requested source area ratio = {requested_source_area_fraction:.3f}")
print(f"actual source area ratio = {source_plane_area_fraction:.3f}")
print(f"overlaps transverse PML = {source_overlaps_transverse_pml}")
print(f"setup = {setup_wall_s:.3f}s, mode solve = {mode_solve_wall_s:.3f}s")

# Show the solved mode metadata, including the effective index.
display(mode_data_to_dataframe(modes))


## Mode Source Plots

Plot the transverse field components of the solved mode source. This is the first check that the source plane is exciting the intended guided mode rather than a badly clipped or off-center field.


In [ ]:
# Plot the magnitude of the dominant TE-like field components on the source plane.
from beamz.analysis.plotting import plot_mode_field_components
fig, axes, neffs = plot_mode_field_components(modes,
    field_names=("Ey", "Ez"),
    mode_indices=(0,),
    val="abs",
    f=freq0,
    figsize=(8, 5),
    show=False,
)
plt.show()


## 3D Design Viewer

Show BEAMZ's static 3D-oriented notebook view for a geometry-level check of the waveguide, mode planes, field-recorder plane, and CPML placement.


In [ ]:
# Display the static 3D-oriented scene inline in the notebook.
if not test_mode:
    sim.view3d(show=True)


## Layout Cross Sections

Show simple 2D slices through the same simulation object. These static slices are useful when the 3D camera angle makes it hard to judge whether the monitor and waveguide center height are aligned.


In [ ]:
# Plot a horizontal slice through the core center and a vertical slice through y=0.
fig, axes = sim.plot(z=0.5 * core_height, y=0.0, show=False)
plt.show()


## Compile and Run

Build the reusable public execution plan, then advance an explicit execution segment. BEAMZ owns JIT compilation internally; the detached `SimulationResults` value owns every monitor acquisition. The reported effective GCUPS includes JIT compilation in the timed call.


In [ ]:
# Build the immutable execution plan through the supported public API.
compile_start = time.perf_counter()
program = sim.compile(num_steps=timed_steps)
compile_setup_wall_s = time.perf_counter() - compile_start

# Execute from a fresh internal state and return detached results.
run_start = time.perf_counter()
sim_data = sim.advance(
    num_steps=timed_steps,
    progress=progress,
).results
compile_and_run_wall_s = time.perf_counter() - run_start
assert sim_data is not None
raw_field_frames = int(sim_data.monitor("ey_slice").field_steps.size)

# This effective throughput intentionally includes first-call JIT compilation.
effective_gcups = cells * timed_steps / compile_and_run_wall_s / 1e9
timing = {
    "setup_wall_s": setup_wall_s,
    "mode_solve_wall_s": mode_solve_wall_s,
    "compile_setup_wall_s": compile_setup_wall_s,
    "compile_and_run_wall_s": compile_and_run_wall_s,
    "cells": cells,
    "timed_steps": timed_steps,
    "cell_updates": int(cells * timed_steps),
    "raw_field_record_interval": raw_field_record_interval,
    "raw_field_frames": raw_field_frames,
    "effective_gcups_including_jit": effective_gcups,
}

# Print machine-readable and human-readable timing summaries.
print(json.dumps(timing, indent=2))
print(f"effective GCUPS including JIT = {effective_gcups:.3f}")


## Modal S-Parameter Diagnostics

Use the downstream source-side mode monitor as the incident reference plane, then plot both modal wave directions at both source-side monitors. This separates backward source injection behind the source from backward guided power returning from the right-side CPML.


In [ ]:
# Canonical ports derive modal wave selectors from their plane and direction.
source_port = bz.Port(
    center=source_back_monitor.center,
    size=source_back_monitor.size,
    name="source_back",
    direction="+",
    mode_spec=mode_spec,
)
forward_port = bz.Port(
    center=source_forward_monitor.center,
    size=source_forward_monitor.size,
    name="source_forward",
    direction="-",
    mode_spec=mode_spec,
)
modal_ports = [source_port, forward_port]

sparam_result = s_parameters(
    sim_data,
    source_port=source_port,
    ports=modal_ports,
    output_ports=modal_ports,
    frequencies=sparam_freqs,
    min_incident_db=-45.0,
)
waves = sparam_result.diagnostics["waves"]
source_incident_selector = sparam_result.diagnostics["monitor_flux_checks"]["source_back"]["incident_wave"]
a_incident = np.asarray(waves["source_back"][f"a_{source_incident_selector}"], dtype=np.complex128)
valid = np.asarray(sparam_result.diagnostics["valid_mask"], dtype=bool)

def safe_ratio(numerator, denominator, floor=1e-18):
    numerator = np.asarray(numerator, dtype=np.complex128)
    denominator = np.asarray(denominator, dtype=np.complex128)
    ratio = np.zeros(np.broadcast_shapes(numerator.shape, denominator.shape), dtype=np.complex128)
    return np.divide(numerator, denominator, out=ratio, where=np.abs(denominator) >= floor)

def amp_db(values, floor=1e-12):
    return 20.0 * np.log10(np.maximum(np.abs(values), floor))

# For x-directed 3D modal projections in BeamZ, minus is the wave moving in
# each Port direction determines which modal wave is incident.
modal_traces = {
    "back monitor: -x wave toward left CPML": safe_ratio(waves["source_back"]["a_plus"], a_incident),
    "back monitor: +x wave toward source": safe_ratio(waves["source_back"]["a_minus"], a_incident),
    "front monitor: +x wave toward right CPML": safe_ratio(waves["source_forward"]["a_plus"], a_incident),
    "front monitor: -x wave returning from right CPML": safe_ratio(waves["source_forward"]["a_minus"], a_incident),
}

fig, ax = plt.subplots(figsize=(8.2, 4.8))
styles = ["s-", "s--", "o-", "o--"]
for (label, values), style in zip(modal_traces.items(), styles):
    ax.plot(sparam_wavelengths_um, amp_db(values), style, label=label)
if not np.all(valid):
    ax.scatter(sparam_wavelengths_um[~valid], amp_db(a_incident)[~valid], marker="x", color="k", label="low incident bin")
ax.axhline(-40.0, color="#666666", lw=1, ls="--", label="-40 dB")
ax.set_xlabel("Wavelength (um)")
ax.set_ylabel("Modal amplitude relative to incident reference (dB)")
ax.set_title("Two-sided source-region modal diagnostics")
ax.grid(True, color="#e6e9ef")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(loc="best")
fig.tight_layout()
plt.show()

center_idx = int(np.argmin(np.abs(sparam_freqs - freq0)))
print(f"center wavelength = {sparam_wavelengths_um[center_idx]:.4f} um")
print(f"incident reference component = a_incident_{source_incident_selector}")
for label, values in modal_traces.items():
    print(f"{label} = {amp_db(values)[center_idx]:.2f} dB")
print(f"valid frequency bins = {int(np.count_nonzero(valid))}/{valid.size}")
sparam_table = {
    "wavelength_um": sparam_wavelengths_um,
    "back_monitor_minus_x_db": amp_db(modal_traces["back monitor: -x wave toward left CPML"]),
    "back_monitor_plus_x_db": amp_db(modal_traces["back monitor: +x wave toward source"]),
    "front_monitor_plus_x_db": amp_db(modal_traces["front monitor: +x wave toward right CPML"]),
    "front_monitor_minus_x_db": amp_db(modal_traces["front monitor: -x wave returning from right CPML"]),
    "valid": valid,
}
display(sparam_table)


## Final Ey Field Slice

Plot the strongest `Ey` time-domain frame captured by the named `FieldRecorder` on `z = core_height / 2`. The frame is selected by RMS field strength near the waveguide core, so the plot shows the launched pulse while it is actually present in the simulation rather than the near-zero final timestep. This is raw field-recorder data, not a DFT field-monitor result.


In [ ]:
# Convert the named field-recorder result to labeled data.
field_result = sim_data.monitor("ey_slice")
ey_frames_xy = sim_data.to_xarray()["Ey"]
ey_values_all = np.asarray(ey_frames_xy) * 1e-6  # V/m -> V/um
if ey_values_all.ndim == 2:
    ey_values_all = ey_values_all[np.newaxis, ...]

actual_field_z_rel = field_z_rel
x_um = (np.asarray(ey_frames_xy.coords["x"]) - sim.coordinate_offset[0]) / um
y_um = (np.asarray(ey_frames_xy.coords["y"]) - sim.coordinate_offset[1]) / um
dx_um = float(np.mean(np.diff(x_um))) if x_um.size > 1 else sim.resolution / um
dy_um = float(np.mean(np.diff(y_um))) if y_um.size > 1 else sim.resolution / um
extent = (
    float(x_um[0] - 0.5 * dx_um),
    float(x_um[-1] + 0.5 * dx_um),
    float(y_um[0] - 0.5 * dy_um),
    float(y_um[-1] + 0.5 * dy_um),
)

core_metric_half_width_um = 0.5 * core_width / um + 0.20
guide_rows = np.abs(y_um) <= core_metric_half_width_um
metric_values = ey_values_all[:, guide_rows, :] if np.any(guide_rows) else ey_values_all
frame_metric = np.sqrt(np.nanmean(metric_values**2, axis=(1, 2)))
frame_idx = int(np.nanargmax(frame_metric)) if frame_metric.size else 0
ey_values = ey_values_all[frame_idx]

field_step = int(field_result.field_steps[frame_idx])
field_time_fs = float(field_result.field_times[frame_idx]) * 1e15

vmax = np.nanpercentile(np.abs(ey_values), 99.5)
vmax = vmax if np.isfinite(vmax) and vmax > 0 else 1.0

fig, ax = plt.subplots(figsize=(7.2, 4.8))
im = ax.imshow(
    ey_values,
    origin="lower",
    extent=extent,
    cmap="RdBu",
    vmin=-vmax,
    vmax=vmax,
    aspect="equal",
    interpolation="nearest",
)
ax.axhspan(-0.5 * core_width / um, 0.5 * core_width / um, color="#888888", alpha=0.45, lw=0)
fig.colorbar(im, ax=ax, label="Ey (V/um)")
title = f"raw Ey at z={actual_field_z_rel:.3f} um, frame {frame_idx + 1}/{ey_values_all.shape[0]}"
if field_step is not None:
    title += f", step={field_step}"
if field_time_fs is not None:
    title += f", t={field_time_fs:.1f} fs"
ax.set_title(title)
ax.set_xlabel("x (um)")
ax.set_ylabel("y (um)")
plt.show()


## Timing Summary

Summarize setup, mode solve, public compile setup, and the first compiled run in one compact horizontal bar chart. The title reports effective throughput including JIT compilation.


In [ ]:
# Prepare the timing categories in the order they occurred.
labels = ["setup", "mode solve", "compile setup", "compile + run"]
values = [
    timing["setup_wall_s"],
    timing["mode_solve_wall_s"],
    timing["compile_setup_wall_s"],
    timing["compile_and_run_wall_s"],
]
colors = ["#8ecae6", "#a3b18a", "#ffb703", "#2a9fb8"]

# Draw a horizontal timing chart with labels at the end of each bar.
fig, ax = plt.subplots(figsize=(7.5, 3.8))
y = np.arange(len(labels))
ax.barh(y, values, color=colors, height=0.55)
ax.set_yticks(y, labels)
ax.invert_yaxis()
ax.set_xlabel("Seconds")
ax.grid(axis="x", color="#e6e9ef")
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
for yi, value in zip(y, values):
    text = f"{value:.2f}s" if value < 10 else f"{value:.1f}s"
    ax.annotate(text, (value, yi), xytext=(6, 0), textcoords="offset points", va="center")
ax.set_title(f"Effective throughput including JIT: {effective_gcups:.3f} GCUPS")
fig.tight_layout()
plt.show()
